# Sentinel-2 Surface Reflectance

**Living Earth - Cube in a Box Demo Series**

---

*   **Objective:** Load, mask Sentinel-2 Level 2A surface reflectance data, compute time statistics and save them as COGs.
*   **Products used:**
    * `s2_l2a` — [Planetary Computer](https://planetarycomputer.microsoft.com/dataset/sentinel-2-l2a) (Explorer: `http://<your-domain>/explorer/products/s2_l2a`)
    * `s2_l2a_cdse` — [Copernicus Data Space](https://stac.dataspace.copernicus.eu/v1/collections/sentinel-2-l2a) (Explorer: `http://<your-domain>/explorer/products/s2_l2a_cdse`)

---

## Choosing a data source

**Planetary Computer (`s2_l2a`) — Pros**
* Assets as **Cloud Optimized GeoTIFFs (COG)** — efficient partial reads over HTTPS
* **No credentials required**; assets signed over HTTPS
* Supports more Dask workers → **faster** `compute()` on multi-core machines

**Planetary Computer — Cons**
* Third-party STAC mirror (Microsoft), not the official Copernicus distribution

**Copernicus Data Space (`s2_l2a_cdse`) — Pros**
* **Official** ESA/Copernicus archive and STAC catalog
* **Native** JPEG 2000 (JP2) tiles — the standard ESA distribution format
* Preferred when you need the **authoritative European data path** or CDSE-only coverage

**Copernicus Data Space — Cons**
* **CDSE S3 credentials must be configured by your administrator**; when enabled, they are provided automatically in Jupyter and the Explorer
* Native JP2 format are **slower to decode** than COG
* S3 **rate limits** → notebook recreates Dask with one worker (**slower loads**)
* **CDSE S3 endpoint pinning can block** unrelated `s3://` reads in the same session

---

## Background

The Sentinel-2 program provides global imagery in thirteen spectral bands at 10m-60m resolution and a revisit time of approximately five days. This dataset represents the global Sentinel-2 archive, from 2016 to the present, processed to L2A (bottom-of-atmosphere) using Sen2Cor.
## Description

This notebook introduces loading Sentinel-2 data, handling multiple resolutions, applying cloud masking using the SCL band, computing time based statistics and save them as COGs (Cloud Optimized Geotiff).

***

In [ ]:
import sys
sys.path.insert(1, './utils/')

In [ ]:
# reload module before executing code
%load_ext autoreload
%autoreload 2

import datacube
import os
import psutil
import rioxarray
import time

import pandas as pd
import xarray as xr
    
from dask.distributed import Client
from IPython.display import HTML, display
from matplotlib import colors
from odc.geo import BoundingBox

from utils.deafrica_plotting import rgb
from utils.le_dc import (
    configure_dask_for_product,
    get_patch_url,
    get_product_bbox,
    recommended_dask_workers,
)
from utils.le_mapping import bbox_to_polygon, display_crosshair, get_utm_epsg_code, MapHandler
from utils.le_masking import scl_mask
from utils.le_tools import style_output_cells

### Connect to the datacube

In [ ]:
dc = datacube.Datacube(app='Sentinel_2')

### Dask dashboard  
In order to optimise usage of your hardware it is recommended to run the next cells to configure Daskerisation and access Dask dashboard, which will allow you to monitor your hardware usage as well as Dask computation progress.

> **Note:** Run the imports cell before starting the Dask client. That sets up Copernicus URL signing and `PYTHONPATH` for worker processes.
>
> For **Copernicus Data Space** (`s2_l2a_cdse`), the notebook recreates the Dask client with a single worker after you select the product, to mitigate CDSE S3 rate limits (HTTP 429) during `compute()`. This is the **main performance trade-off** when choosing CDSE over Planetary Computer (see intro for full pros/cons).

![](./figures/dask_dashboard.png)

In [ ]:
# Get the available cpu(s) number and memory in gigabytes (to properly configure Dask client in follwoing cell)

print(f"Available cpu(s): {psutil.cpu_count()}")

available_memory = psutil.virtual_memory().available
available_memory_gb = available_memory / (1024 ** 3)
print(f"Available memory: {available_memory_gb:.2f} GB")

In [ ]:
try:
    client.dashboard_link
except:
    client = Client(n_workers=4, memory_limit='4GB')  # memory_limit PER WORKER !
client

## Load Sentinel-2 data from the datacube

We will load **Sentinel-2** data using two methods.
Firstly, we will use [dc.load()](../Beginners_guide/03_Loading_data.ipynb) to return a time series of satellite images.
Secondly, we will load a time series using the [load_ard()](../Frequently_used_code/Using_load_ard.ipynb) function, which is a wrapper function around the `dc.load` module.
This function will load all the images from Sentinel-2 and then apply a cloud/pixel-quality mask.
The returned `xarray.Dataset` will contain analysis ready images with the cloudy and invalid pixels masked out.

You can change any of the parameters in the `query` object below to adjust the location, time, projection, or spatial resolution of the returned datasets.
To learn more about querying, refer to the Beginner's guide notebook on [loading data](../Beginners_guide/03_Loading_data.ipynb).

Sentinel-2 data is stored on file with a range of different coordinate reference systems or CRS (i.e. multiple UTM zones). 
The different satellite bands also have different resolutions (10 m, 20 m and 60 m). 
Because of this, all Sentinel-2 queries need to include the following two query parameters:

* `output_crs`: This sets a consistent CRS that all Sentinel-2 data will be reprojected to, regardless of the UTM zone the individual image is stored in.
* `resolution`: This sets the resolution that all Sentinel-2 images will be resampled to. 

> **Note:** Be aware that setting `resolution` to the highest available resolution (i.e. `10`) will downsample the coarser resolution 20 m and 60 m bands, which may introduce unintended artefacts into your analysis.
It is typically best practice to set `resolution` to match the lowest resolution band being analysed. For example, if your analysis uses both 10 m and 20 m resolution bands, set `"resolution": 20`.

In [ ]:
# Check if default bbox is contained within the datacube
# and allow user to draw bbox if not.

# Sentinel-2 L2A is available from two indexed products (same workflow below):
#   s2_l2a (Fast)      — Planetary Computer (default; COG over HTTPS; more Dask workers)
#   s2_l2a_cdse (Slow) — Copernicus Data Space (JP2 tiles; CDSE creds in Jupyter; 1 Dask worker)
# See the intro cell for pros/cons. Use whichever product exists in your datacube.
product = 's2_l2a'  # or 's2_l2a_cdse'

# configure a default bounding box and visualize it
lat, lon = 22.821, 28.518
buffer = 0.05
default_bbox = (lon - buffer, lat - buffer, lon + buffer, lat + buffer)

product_bbox = get_product_bbox(dc, product, split_size=10, stability_threshold=4)

is_contained =(default_bbox[0] >= product_bbox[0] and
               default_bbox[1] >= product_bbox[1] and
               default_bbox[2] <= product_bbox[2] and
               default_bbox[3] <= product_bbox[3]
              )

# Recreate Dask client for the selected product (CDSE needs fewer workers).
try:
    client.close()
except Exception:
    pass

n_workers = recommended_dask_workers(dc, product)
client = Client(n_workers=n_workers, memory_limit='4GB')
configure_dask_for_product(dc, product, client)

cogs_folder = './cogs/'

In [ ]:
# Create an instance of MapHandler
map_handler = MapHandler()
m, drc = map_handler.create_map(vect=[bbox_to_polygon(default_bbox), bbox_to_polygon(product_bbox)],
                               draw_rect=True)
display(m)

# append crosshair
time.sleep(2)  # make sure m is fully displayed
display_crosshair()

In [ ]:
# Warn in case of full AoI
aoi_poly = map_handler.aoi_tupple

if aoi_poly is None:
    aoi_poly = tuple(default_bbox)
    if not is_contained:
        style_output_cells('salmon', border_color='red', border_width='2px')
        print('The area of interest polygon is located outside of the product extent.' + \
              '\nPlease draw a new area of interest in the previous cell.')
    else:
        # When is_contained is True and no polygon drawn - this is actually OK!
        style_output_cells()
        print('Default area of interest is contained within the product extent, but you can still draw another one in the previous cell.')
else:
    # A polygon was drawn
    style_output_cells()
    print('Custom area of interest polygon has been created.')

In [ ]:
# get EPSG code for the center of the AoI

epsg_code = get_utm_epsg_code((aoi_poly[1] + aoi_poly[3]) / 2, (aoi_poly[0] + aoi_poly[2]) / 2)

In [ ]:
times = [ds.time.begin for ds in dc.find_datasets(product=product)]

start_date = min(times)
end_date = max(times)

In [ ]:
import ipywidgets as widgets
start_date = widgets.DatePicker(description='Start date',
                                value = min(times).date(),
                                disabled=False)
end_date = widgets.DatePicker(description='End date',
                                value = max(times).date(),
                                disabled=False)
display(widgets.Label('IF REQUIRED define time period (cannot be outside of the initial displayed time) and run the next cell:'),
        widgets.HBox([start_date, end_date]))

In [ ]:
query = {
    'time': (start_date.value.strftime('%Y-%m-%d'), end_date.value.strftime('%Y-%m-%d')),
    'x': (aoi_poly[0], aoi_poly[2]),
    'y': (aoi_poly[1], aoi_poly[3]),
    'output_crs': f"EPSG:{epsg_code}",
    'resolution':20,
    'group_by': "solar_day",
}

### Load Sentinel-2 using `dc.load()`

The **Sentinel-2** products are:

* `s2_l2a` — Planetary Computer
* `s2_l2a_cdse` — Copernicus Data Space

Both contain images from Sentinel-2 sensors S2A and S2B.

> **Source comparison:** The band aliases used in this notebook (`blue`, `red`, `nir`, `SCL`, etc.) work for both products. Planetary Computer serves COG GeoTIFFs; Copernicus Data Space serves native JP2 tiles. CDSE credentials, `get_patch_url`, and Dask worker count are handled automatically by `utils.le_dc` when your administrator has configured CDSE access in Jupyter and the Explorer. See the intro for full pros/cons.

We will now load in a time-series of satellite images from only Sentinel-2

> **Note:** In this example, we include the `dask_chunks={}` parameter in order to "lazy-load" the data. The returned array will contain information about the data without directly loading it, which saves time in this instance. For more information on this, see the Beginner's guide notebook on [parallel processing with Dask](../Beginners_guide/06_Parallel_processing_with_dask.ipynb).

In [ ]:
#load data
lazy_ds = dc.load(product=product,
             dask_chunks={},
             patch_url=get_patch_url(dc, product),
             **query)

print(lazy_ds)

The returned dataset contains all of the bands available for Sentinel-2. These bands are named using the datasets native band-names.  We can also specify which bands to return using band-name `aliases`, such as `red` or `nir`. We do this using the parameter `measurements`.

In [ ]:
bands = ['blue', 'green', 'red', 'nir', 'swir_1', 'SCL']

lazy_ds = dc.load(product=product,
                  measurements=bands,
                  dask_chunks={},
                  patch_url=get_patch_url(dc, product),
                  **query)

print(lazy_ds)

So far only metadata was accessed (which explains why loading was fast), which is called "lazy" loading. Now we can load data for true (as the rgb function will be used several time and will do it everytime it is run). If you opened the Dask client link above you will be able to monitor activity, and possibly adapt dask_chunks parameters for better efficiency and stability.

In [ ]:
%%time

# customized chunking is less buggy when performed after loading !!!
lazy_ds = lazy_ds.chunk({"x": 512, "y": 512, "time": 1})

ds = lazy_ds.compute()
print(ds)

## Remove pixels considered as nodata and convert to SR

In [ ]:
valid_cats = [4, 5, 6, 7]  # 11: snow ???
mask = scl_mask(ds.SCL, valid_cats)
ds = ds.drop_vars('SCL')  # Not needed anymore
ds = ds.where(ds <= 10000) / 10000  # Convert DN to SR and remove saturated pixels
ds = ds.where(mask)
ds = ds.dropna('time', how='all')  # Remove empty scenes

In [ ]:
# Plot a false color composite of all scenes

rgb(ds, bands=['nir', 'red', 'green'], col="time", col_wrap=4)

In [ ]:
# Plot a selected scene (by default robust argument is True and might return error if set to False,
# but an equivalent option is to set percentile_stretch to (0,1), keeping default robust value).

rgb(ds, bands=['nir', 'red', 'green'], index=0, percentile_stretch=(0, 1))

In [ ]:
# Plot a selection of scenes (mixing time)

rgb(ds, bands=['nir', 'red', 'green'], index=[0, 2, 1, -1])

## Compute NDVI

In [ ]:
ndvi = (ds.nir - ds.red) / (ds.nir + ds.red)

# Define a vegetation colormap (blue, white

vegetation_cm = colors.LinearSegmentedColormap.from_list(
    'vegetation',
    ['#051852', '#051852', '#051852', '#051852',   # deep blue
     '#ffffff',                                    # white
     '#bca675', '#82b402', '#187c02', '#003b00'],  # brown to green
    N=256
)
ndvi.plot(col='time', col_wrap=4, cmap = vegetation_cm, vmin = -1, vmax = 1)

## Time based statistics

As default CiaB installation offer a very limited time range (1 month), let's compute 1 week min, mean and max values.

In [ ]:
ndvi['time'] = pd.to_datetime(ndvi.time.values)
weekly_resampled = ndvi.resample(time='W')

weekly_ds = xr.Dataset({
    'ndvi_mean': weekly_resampled.mean(),
    'ndvi_min':  weekly_resampled.min(),
    'ndvi_max':  weekly_resampled.max()
})

print(weekly_ds)

# Export as COGs (Cloud Optimized Geotiff)

In [ ]:
for i in range(len(weekly_ds.time)):
    date_str = str(weekly_ds.time.values[i])[:10]
    folder_name = os.path.join(cogs_folder, f"week_{date_str}")
    
    os.makedirs(folder_name, exist_ok=True)
    
    for var_name in weekly_ds.data_vars:
        single_band = weekly_ds[var_name].isel(time=i)
        filename = os.path.join(folder_name, f"{var_name}.tif")
        single_band.rio.to_raster(
            filename,
            driver="COG",
            nodata=float('nan') 
        )
    
    print(f"{folder_name}/ created")

***

## Additional information

**License:** The code in this notebook is slighly modified from https://github.com/digitalearthafrica/deafrica-sandbox-notebooks and licensed under the [Apache License, Version 2.0](https://www.apache.org/licenses/LICENSE-2.0).

**Compatible datacube version:**

In [ ]:
print(datacube.__version__)

**Last tested:**

In [ ]:
from datetime import datetime
datetime.today().strftime('%Y-%m-%d')

In [ ]:
!pip freeze